In [1]:
import numpy as np
import cv2 as cv
import json
import os

In [2]:
def crop_rect(image, rect):
    center, size, angle = rect[0:3]
    height, width = image.shape[0:2]

    if int(angle) >= 45:
        angle = angle - 90
        temp = size
        size = (temp[1], temp[0])

    rotation_matrix = cv.getRotationMatrix2D(center, angle, 1)
    center, size = tuple(map(int, center)), tuple(map(int, size))

    image_rot = cv.warpAffine(image, rotation_matrix, (width, height))
    image_crop = cv.getRectSubPix(image_rot, size, center)

    return image_crop

In [3]:
def crop_image(cv_image, annotations, date, time, subset, save_path):
    for crop in annotations:
        try:
            crop_id = crop["id"]
            category = crop["category_id"]

            poly = np.array(crop["segmentation"])
            poly = np.reshape(poly, (-1, 2))
            poly = np.int64(poly)

            rotaded_rect = cv.minAreaRect(poly)

            box = cv.boxPoints(rotaded_rect)
            box = np.int64(box)

            cropped = crop_rect(cv_image, rotaded_rect)

            image_name = f"{date}_{time}#{subset}#{crop_id}"
            subfolder = f"{date}/{'occupied' if category == 1 else 'empty'}"
            print(f"Saving image {image_name} into {save_path}/{subfolder}")

            cv.imwrite(f"{save_path}/{subfolder}/{image_name}.jpg", cropped)
        except:
            print(f"Failed save image {image_name} into {save_path}/{subfolder}")

In [4]:
ROOT_PATH = "/home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLot"
DEST_PATH = "/home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLotSegmented"

In [5]:
with open("/home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLot/pucpr_spots.json", "r") as file:
    pucpr = json.load(file)

with open("/home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLot/ufpr04_spots.json", "r") as file:
    ufpr04 = json.load(file)

with open("/home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLot/ufpr05_spots.json", "r") as file:
    ufpr05 = json.load(file)

In [6]:
annotations_dict = {'pucpr': pucpr["annotations"], 'ufpr04': ufpr04["annotations"], 'ufpr05': ufpr05["annotations"]}

In [7]:
with open("_failed/PKLotSegmented_failed_images.json", "r") as file:
    data = json.load(file)

data = [file for file in data if "2012-10-16" not in file["file_name"]]

In [8]:
for image in data: 
    cv_image = cv.imread(f"{ROOT_PATH}/{image['file_name']}")

    annotations = [annotation for annotation in annotations_dict[image["subset"]] if annotation["image_id"] == image["id"]]
    date = image["file_name"].split("/")[2]
    time = "_".join(image["file_name"].split("/")[3].split(date)[1].split(".")[0].split("_")[1:])
    subset = image["subset"]

    if not os.path.exists(f"{DEST_PATH}/{date}"):
        os.mkdir(f"{DEST_PATH}/{date}")
        os.mkdir(f"{DEST_PATH}/{date}/empty")
        os.mkdir(f"{DEST_PATH}/{date}/occupied")

    crop_image(cv_image, annotations, date, time, subset, DEST_PATH)

Saving image 2012-10-13_07_03_36#pucpr#410866 into /home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLotSegmented/2012-10-13/empty
Saving image 2012-10-13_07_03_36#pucpr#410867 into /home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLotSegmented/2012-10-13/empty
Saving image 2012-10-13_07_03_36#pucpr#410868 into /home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLotSegmented/2012-10-13/empty
Saving image 2012-10-13_07_03_36#pucpr#410869 into /home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLotSegmented/2012-10-13/empty
Saving image 2012-10-13_07_03_36#pucpr#410870 into /home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLotSegmented/2012-10-13/empty
Saving image 2012-10-13_07_03_36#pucpr#410871 into /home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLotSegmented/2012-10-13/empty
Saving image 2012-10-13_07_03_36#pucpr#410872 into /home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLotSegmented/2012-10-13/empty
Saving image 2012-10-13_07_03_36#pucpr#410873 into /home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLotSegmented/2012-10-13/empty


In [13]:
wrong = []
for image in data: 
    cv_image = cv.imread(f"{ROOT_PATH}/{image['file_name']}")

    annotations = [annotation for annotation in annotations_dict[image["subset"]] if annotation["image_id"] == image["id"]]

    for annotation in annotations:
        if annotation['category_id'] != 0 and annotation['category_id'] != 1:
            wrong.append(annotation)

In [ ]:
folders = os.listdir(DEST_PATH)

for folder in folders:
    empty = len(os.listdir(f"{DEST_PATH}/{folder}/empty"))
    occupied = len(os.listdir(f"{DEST_PATH}/{folder}/occupied"))    

    if empty == 0:
        os.rmdir(f"{DEST_PATH}/{folder}/empty")
        print(f"removing {DEST_PATH}/{folder}/empty")

    if occupied == 0:
        os.rmdir(f"{DEST_PATH}/{folder}/occupied")
        print(f"removing {DEST_PATH}/{folder}/occupied")